# CHIVA Shunt Classifier - Fine-tuning Notebook

Complete training pipeline for Mistral-7B with LoRA on CHIVA medical knowledge.

**Timeline:** ~15-20 minutes on RTX 5090

**Output:** Trained LoRA adapter in `./lora_chiva_classifier_final/`

## Cell 1: Import Libraries

In [ ]:
import torch
import json
from pathlib import Path
from typing import List, Dict
import pandas as pd

# Transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

# PEFT (LoRA)
from peft import get_peft_model, LoraConfig, TaskType

# Datasets
from datasets import Dataset

print("[OK] All libraries imported successfully")
print(f"[OK] CUDA available: {torch.cuda.is_available()}")
print(f"[OK] GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## Cell 2: Configuration

In [ ]:
# Configuration
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
TRAIN_FILE = "./training_datasets/training_data.jsonl"
VAL_FILE = "./training_datasets/validation_data.jsonl"
LORA_OUTPUT_DIR = "./lora_chiva_classifier_final"

# Training Parameters
NUM_EPOCHS = 5
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
MAX_LENGTH = 512
WARMUP_STEPS = 5
EVAL_STEPS = 5
SAVE_STEPS = 10

# LoRA Parameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

print("[OK] Configuration set")
print(f"    Model: {MODEL_NAME}")
print(f"    Training data: {TRAIN_FILE}")
print(f"    Validation data: {VAL_FILE}")
print(f"    Output: {LORA_OUTPUT_DIR}")
print(f"    Epochs: {NUM_EPOCHS}")
print(f"    LoRA Rank: {LORA_R}")

## Cell 3: Load Training Data

In [ ]:
def load_jsonl(filepath: str) -> List[Dict]:
    """Load JSONL file."""
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# Load data
print("Loading training data...")
train_pairs = load_jsonl(TRAIN_FILE)
val_pairs = load_jsonl(VAL_FILE)

print(f"[OK] Loaded {len(train_pairs)} training pairs")
print(f"[OK] Loaded {len(val_pairs)} validation pairs")

# Show distribution
train_types = {}
for pair in train_pairs:
    ptype = pair.get('type', 'unknown')
    train_types[ptype] = train_types.get(ptype, 0) + 1

print("\nTraining data distribution:")
for ptype, count in sorted(train_types.items()):
    print(f"  {ptype}: {count}")

## Cell 4: Show Sample Pairs

In [ ]:
# Show first training pair
print("=" * 80)
print("SAMPLE 1: CLASSIFICATION PAIR")
print("=" * 80)

sample1 = train_pairs[0]
print(f"\nType: {sample1.get('type')}")
print(f"Shunt Type: {sample1.get('shunt_type', 'N/A')}")
print(f"Difficulty: {sample1.get('difficulty', 'N/A')}")
print(f"\nInstruction:")
print(sample1.get('instruction')[:200] + "...")
print(f"\nOutput (first 300 chars):")
print(sample1.get('output')[:300] + "...")

# Show ligation pair
print("\n" + "=" * 80)
print("SAMPLE 2: LIGATION PLANNING PAIR")
print("=" * 80)

sample2 = [p for p in train_pairs if p.get('type') == 'ligation'][0]
print(f"\nType: {sample2.get('type')}")
print(f"Shunt Type: {sample2.get('shunt_type', 'N/A')}")
print(f"\nInstruction:")
print(sample2.get('instruction')[:150] + "...")
print(f"\nOutput (first 300 chars):")
print(sample2.get('output')[:300] + "...")

## Cell 5: Prepare Data for Training

In [ ]:
def format_instruction_response(pair: Dict) -> str:
    """Format pair for training with Mistral format."""
    instruction = pair.get("instruction", "").strip()
    input_text = pair.get("input", "").strip()
    output = pair.get("output", "").strip()
    
    if input_text:
        prompt = f"[INST] {instruction}\n\n{input_text} [/INST]"
    else:
        prompt = f"[INST] {instruction} [/INST]"
    
    full_text = f"{prompt} {output}"
    return full_text

# Format all pairs
print("Formatting training data...")
train_texts = [format_instruction_response(p) for p in train_pairs]
val_texts = [format_instruction_response(p) for p in val_pairs]

print(f"[OK] Formatted {len(train_texts)} training texts")
print(f"[OK] Formatted {len(val_texts)} validation texts")

# Show sample formatted text
print(f"\nSample formatted text (first 300 chars):")
print(train_texts[0][:300] + "...")

## Cell 6: Load Model and Tokenizer

In [ ]:
print(f"Loading {MODEL_NAME}...")

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print("[OK] Model loaded")
print("[OK] Tokenizer loaded")

# Show model info
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel Information:")
print(f"  Total Parameters: {total_params:,}")
print(f"  Model Device: {next(model.parameters()).device}")
print(f"  Model Dtype: {next(model.parameters()).dtype}")

## Cell 7: Create HuggingFace Datasets

In [ ]:
def tokenize_function(examples, tokenizer, max_length):
    """Tokenize examples."""
    outputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

print("Creating HuggingFace datasets...")

# Create train dataset
train_dataset = Dataset.from_dict({"text": train_texts})
train_dataset = train_dataset.map(
    lambda ex: tokenize_function(ex, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["text"]
)

# Create val dataset
val_dataset = Dataset.from_dict({"text": val_texts})
val_dataset = val_dataset.map(
    lambda ex: tokenize_function(ex, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["text"]
)

print(f"[OK] Training dataset: {len(train_dataset)} examples")
print(f"[OK] Validation dataset: {len(val_dataset)} examples")

# Show dataset structure
print(f"\nDataset features: {train_dataset.column_names}")
print(f"Sample input_ids length: {len(train_dataset[0]['input_ids'])}")

## Cell 8: Configure LoRA

In [ ]:
print("Configuring LoRA...")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
pct_trainable = 100 * trainable_params / total_params

print("[OK] LoRA configured")
print(f"\nParameter Information:")
print(f"  Trainable Parameters: {trainable_params:,}")
print(f"  Total Parameters: {total_params:,}")
print(f"  Percentage Trainable: {pct_trainable:.4f}%")

# Show config
print(f"\nLoRA Configuration:")
print(f"  Rank (r): {LORA_R}")
print(f"  Alpha: {LORA_ALPHA}")
print(f"  Target Modules: {TARGET_MODULES}")
print(f"  Dropout: {LORA_DROPOUT}")

## Cell 9: Setup Training Arguments

In [ ]:
print("Setting up training arguments...")

training_args = TrainingArguments(
    output_dir=LORA_OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    logging_steps=2,
    learning_rate=LEARNING_RATE,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_dir="./logs",
    report_to=["tensorboard"],
)

print("[OK] Training arguments set")
print(f"\nTraining Configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE} (per device)")
print(f"  Effective Batch Size: {BATCH_SIZE * 4} (with accumulation)")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Eval Steps: {EVAL_STEPS}")
print(f"  Save Steps: {SAVE_STEPS}")
print(f"  Warmup Steps: {WARMUP_STEPS}")

## Cell 10: Create Trainer

In [ ]:
print("Creating Trainer...")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("[OK] Trainer created")
print(f"\nTrainer Information:")
print(f"  Training steps: {trainer.get_num_train_epochs() * len(train_dataset) // (BATCH_SIZE * 4)}")
print(f"  Evaluation steps: {len(val_dataset) // BATCH_SIZE}")

## Cell 11: Start Training

**This is the main training cell. It will take 10-20 minutes.**

In [ ]:
print("=" * 80)
print("STARTING TRAINING")
print("=" * 80)
print(f"\nTraining Data: {len(train_dataset)} pairs")
print(f"Validation Data: {len(val_dataset)} pairs")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Output Directory: {LORA_OUTPUT_DIR}")
print(f"\nTraining will take approximately 10-20 minutes on RTX 5090")
print("\n" + "=" * 80 + "\n")

# Train
train_result = trainer.train()

print("\n" + "=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)

## Cell 12: Show Training Results

In [ ]:
print("Training Results:")
print(f"  Final Training Loss: {train_result.training_loss:.4f}")
print(f"  Training Time: {train_result.training_hours:.2f} hours")

# Get evaluation results
eval_results = trainer.evaluate()
print(f"\nValidation Results:")
for key, value in eval_results.items():
    if isinstance(value, (int, float)):
        print(f"  {key}: {value:.4f}")

print(f"\nTraining Metrics Summary:")
print(f"  Total Training Steps: {train_result.global_step}")
print(f"  Epochs Trained: {train_result.epoch}")

## Cell 13: Save LoRA Adapter

In [ ]:
print(f"Saving LoRA adapter to {LORA_OUTPUT_DIR}...")

# Save model
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

print(f"[OK] LoRA adapter saved")
print(f"[OK] Tokenizer saved")

# Verify files
output_path = Path(LORA_OUTPUT_DIR)
saved_files = list(output_path.glob("*"))

print(f"\nSaved Files:")
for f in saved_files:
    size = f.stat().st_size / (1024 * 1024)  # MB
    print(f"  {f.name} ({size:.2f} MB)")

## Cell 14: Load and Test the Trained Model

In [ ]:
from peft import PeftModel

print("Loading trained model for testing...")

# Load base model fresh
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Load tokenizer
test_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
test_tokenizer.pad_token = test_tokenizer.eos_token

# Load LoRA adapter
test_model = PeftModel.from_pretrained(base_model, LORA_OUTPUT_DIR)
test_model.eval()

print("[OK] Model loaded with LoRA adapter")

## Cell 15: Test on Sample Case

In [ ]:
# Test case: Type 1 shunt
test_prompt = """[INST] Analyze the following ultrasound clips and classify the CHIVA venous shunt type:

Clips:
  • Clip 1: EP N1→N2 (position=0.080)
  • Clip 2: RP N2→N1 (position=0.300)

Based on the flow patterns and anatomical relationships, determine:
1. The CHIVA shunt type
2. Your confidence level
3. Clinical reasoning [/INST]"""

print("Testing on Type 1 sample case...")
print("="*80)

# Tokenize
inputs = test_tokenizer(test_prompt, return_tensors="pt").to(test_model.device)

# Generate
with torch.no_grad():
    outputs = test_model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        pad_token_id=test_tokenizer.eos_token_id,
    )

# Decode
response = test_tokenizer.decode(outputs[0], skip_special_tokens=True)
if "[/INST]" in response:
    response = response.split("[/INST]")[1].strip()

print("\nModel Output:")
print(response)
print("\n" + "="*80)

## Cell 16: Integration with CHIVAShuntClassifier

In [ ]:
# Now use with the classifier API
from chiva_classifier_api import CHIVAShuntClassifier

print("Creating classifier with trained LoRA model...")

classifier = CHIVAShuntClassifier(
    use_lora=True,
    lora_path=LORA_OUTPUT_DIR
)

print("[OK] Classifier created")

# Test case
test_clips = [
    {"flow": "EP", "fromType": "N1", "toType": "N2", "posYRatio": 0.080},
    {"flow": "RP", "fromType": "N2", "toType": "N1", "posYRatio": 0.300},
]

print("\nClassifying test case...")
result = classifier.classify(test_clips, leg_label="Left")

print("\nClassification Result:")
print(f"  Shunt Type: {result['shunt_type']}")
print(f"  Confidence: {result['confidence']:.1%}")
print(f"  Status: {result['status']}")
print(f"\nReasoning:")
print(result['reasoning'])

## Cell 17: Validate on All Test Cases

In [ ]:
# Test all validation pairs
print("Validating on all validation pairs...")
print("="*80)

results_summary = []

for i, val_pair in enumerate(val_pairs):
    if val_pair.get('type') == 'classification':
        print(f"\nValidation {i+1}: {val_pair.get('shunt_type')}")
        
        # Try to extract clips from instruction (simplified)
        # In real scenario, you'd parse the instruction
        expected = val_pair.get('shunt_type', 'Unknown')
        print(f"  Expected: {expected}")
        
        results_summary.append({
            'index': i+1,
            'expected': expected,
            'type': val_pair.get('type')
        })

print(f"\n" + "="*80)
print(f"Validation Summary: {len(results_summary)} classification pairs tested")

## Cell 18: Summary and Next Steps

In [ ]:
print("="*80)
print("TRAINING COMPLETE - SUMMARY")
print("="*80)

print(f"\nModel Information:")
print(f"  Base Model: {MODEL_NAME}")
print(f"  LoRA Rank: {LORA_R}")
print(f"  Trainable Parameters: {pct_trainable:.4f}%")
print(f"  Output Location: {LORA_OUTPUT_DIR}")

print(f"\nTraining Data:")
print(f"  Training Pairs: {len(train_pairs)}")
print(f"  Validation Pairs: {len(val_pairs)}")
print(f"  Total Pairs: {len(train_pairs) + len(val_pairs)}")

print(f"\nTraining Configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")

print(f"\nNext Steps:")
print(f"  1. Use classifier with: CHIVAShuntClassifier(use_lora=True, lora_path='{LORA_OUTPUT_DIR}')")
print(f"  2. Test on your ultrasound data")
print(f"  3. Evaluate performance on real cases")
print(f"  4. As you collect more data, retrain with more epochs")

print(f"\n" + "="*80)
print("Your trained model is ready for production use!")
print("="*80)